# 回測結果分析
分析 `logic/auto_trading/output/trade_log.csv` 的交易內容。

# 目錄

- [1. 多空方向分析](#section-1)
- [2. 出場原因分析](#section-2)
- [3. 持有天數分佈](#section-3)
- [4. 年度損益分析](#section-4)
- [5. TAIEX 大盤環境 vs 交易結果](#section-5)
- [6. 贏家 vs 輸家 持有天數比較](#section-6)
- [7. Phase1 vs Phase2 出場比例](#section-7)
- [8. 最大虧損前 20 筆](#section-8)
- [9. 持有超過 30 天但損益 <= 0（白持的交易）](#section-9)
- [10. 個股交易分析](#section-10)


In [ ]:
%%time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('font', family='Microsoft JhengHei')

TRADE_LOG = 'logic/auto_trading/output/trade_log.csv'
TAIEX_CSV = 'taiex.csv'

df = pd.read_csv(TRADE_LOG, parse_dates=['entry_date', 'exit_date'])
print(f'總交易筆數：{len(df)}')
df.head()

<a id="section-1"></a>
## 1. 多空方向分析

In [ ]:
%%time
for d, g in df.groupby('direction'):
    wins = g[g.pnl_net > 0].pnl_net.sum()
    loss = g[g.pnl_net < 0].pnl_net.abs().sum()
    pf   = wins / loss if loss > 0 else float('inf')
    wr   = (g.pnl_net > 0).mean()
    print(f"{d:6s}  筆數={len(g):4d}  勝率={wr:.1%}  "
          f"平均獲利={g[g.pnl_net>0].pnl_net.mean():.0f}  "
          f"平均虧損={g[g.pnl_net<0].pnl_net.mean():.0f}  PF={pf:.3f}")

<a id="section-2"></a>
## 2. 出場原因分析

In [ ]:
%%time
summary = df.groupby('exit_reason').agg(
    筆數=('pnl_net', 'count'),
    勝率=('pnl_net', lambda x: (x > 0).mean()),
    平均損益=('pnl_net', 'mean'),
    總損益=('pnl_net', 'sum'),
).round(1)
summary['勝率'] = summary['勝率'].map('{:.1%}'.format)
print(summary.to_string())

<a id="section-3"></a>
## 3. 持有天數分佈

In [ ]:
%%time
print(df['hold_days'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['hold_days'].hist(bins=40, ax=axes[0], color='#1565C0', alpha=0.7)
axes[0].set_title('持有天數分佈')
axes[0].set_xlabel('天數')

df.groupby('direction')['hold_days'].mean().plot(
    kind='bar', ax=axes[1], color=['#C62828', '#1565C0']
)
axes[1].set_title('多空平均持有天數')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

<a id="section-4"></a>
## 4. 年度損益分析

In [ ]:
%%time
df['year'] = df['exit_date'].dt.year
yearly = df.groupby('year').agg(
    筆數=('pnl_net', 'count'),
    總損益=('pnl_net', 'sum'),
    勝率=('pnl_net', lambda x: (x > 0).mean()),
).round(1)
yearly['勝率'] = yearly['勝率'].map('{:.1%}'.format)
print(yearly.to_string())

df.groupby('year')['pnl_net'].sum().plot(
    kind='bar', figsize=(12, 4), color='#1565C0', alpha=0.8
)
plt.axhline(0, color='red', linewidth=0.8)
plt.title('年度總損益（元）')
plt.ylabel('損益')
plt.tight_layout()
plt.show()

<a id="section-5"></a>
## 5. TAIEX 大盤環境 vs 交易結果

In [ ]:
%%time
taiex = pd.read_csv(TAIEX_CSV, index_col='Date', parse_dates=True)
taiex['EMA_200'] = taiex['Close'].ewm(span=200, adjust=False).mean()
taiex['regime']  = (taiex['Close'] > taiex['EMA_200']).map({True: 'long', False: 'short'})

regime_map = taiex['regime'].to_dict()
df['regime'] = df['entry_date'].map(regime_map)

print('大盤環境分佈（進場日）：')
print(df['regime'].value_counts())
print()

for r, g in df.groupby('regime'):
    wins = g[g.pnl_net > 0].pnl_net.sum()
    loss = g[g.pnl_net < 0].pnl_net.abs().sum()
    pf   = wins / loss if loss > 0 else float('inf')
    print(f"regime={r:6s}  筆數={len(g):4d}  勝率={(g.pnl_net>0).mean():.1%}  "
          f"PF={pf:.3f}  總損益={g.pnl_net.sum():.0f}")

<a id="section-6"></a>
## 6. 贏家 vs 輸家 持有天數比較

In [ ]:
%%time
wins_df   = df[df.pnl_net > 0]
losses_df = df[df.pnl_net <= 0]

print(f"贏家平均持有天數：{wins_df.hold_days.mean():.1f} 天  中位數：{wins_df.hold_days.median():.0f} 天")
print(f"輸家平均持有天數：{losses_df.hold_days.mean():.1f} 天  中位數：{losses_df.hold_days.median():.0f} 天")
print()
print("結論：若贏家持有天數 < 輸家，代表贏的太早出場、輸的拖太久")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
wins_df.hold_days.hist(bins=30, ax=axes[0], color='#2E7D32', alpha=0.7)
axes[0].set_title('贏家持有天數')
axes[0].set_xlabel('天數')
losses_df.hold_days.hist(bins=30, ax=axes[1], color='#C62828', alpha=0.7)
axes[1].set_title('輸家持有天數')
axes[1].set_xlabel('天數')
plt.tight_layout()
plt.show()

<a id="section-7"></a>
## 7. Phase1 vs Phase2 出場比例

In [ ]:
%%time
for reason, g in df.groupby('exit_reason'):
    wins = g[g.pnl_net > 0].pnl_net.sum()
    loss = g[g.pnl_net < 0].pnl_net.abs().sum()
    pf   = wins / loss if loss > 0 else float('inf')
    print(f"{reason:15s}  筆數={len(g):4d} ({len(g)/len(df):.0%})  "
          f"勝率={(g.pnl_net>0).mean():.1%}  "
          f"平均持有={g.hold_days.mean():.1f}天  "
          f"PF={pf:.3f}  總損益={g.pnl_net.sum():.0f}")

print()
print("phase1_stop  = 進場後沒獲利，被 10 日低點停損砍掉（假突破）")
print("phase2_trail = 漲上去有獲利，最後被 ATR 追蹤停損拉出")

<a id="section-8"></a>
## 8. 最大虧損前 20 筆

In [ ]:
%%time
cols = ['ticker', 'direction', 'entry_date', 'exit_date', 'hold_days',
        'adj_entry_price', 'adj_exit_price', 'pnl_net', 'exit_reason']
worst = df.nsmallest(20, 'pnl_net')[cols]
print(worst.to_string(index=False))

<a id="section-9"></a>
## 9. 持有超過 30 天但損益 <= 0（白持的交易）

In [ ]:
%%time
long_losers = df[(df.hold_days >= 30) & (df.pnl_net <= 0)].sort_values('pnl_net')
print(f"持有 >= 30 天且虧損：{len(long_losers)} 筆  總虧損：{long_losers.pnl_net.sum():.0f} 元")
print()
print(long_losers[cols].to_string(index=False))

<a id="section-10"></a>
## 10. 個股交易分析

In [ ]:
%%time
import os

def plot_trade(ticker, entry_date, df, stocks_dir='stocks_full', window=30):
    """
    繪製單筆交易的個股走勢圖。
    ticker     : 股票代碼 (str)，例如 '1905'
    entry_date : 進場日期 (str 'YYYY-MM-DD' 或 datetime)
    df         : trade_log DataFrame（由 cell-setup 載入）
    window     : 進場前 / 出場後各顯示幾個交易日（預設 30）
    """
    ticker = str(ticker)
    trade = df[(df.ticker == ticker) & (df.entry_date == pd.Timestamp(entry_date))]
    if trade.empty:
        print(f'找不到 {ticker} {entry_date} 的交易紀錄')
        return
    trade = trade.iloc[0]

    csv_path = os.path.join(stocks_dir, f'{ticker}.csv')
    if not os.path.exists(csv_path):
        print(f'找不到個股資料：{csv_path}')
        return

    stock = pd.read_csv(csv_path, parse_dates=['日期'])
    stock = stock.rename(columns={
        '日期': 'Date', '開盤價': 'Open', '最高價': 'High',
        '最低價': 'Low',  '收盤價': 'Close', '成交股數': 'Volume'
    })
    for col in ['Open', 'High', 'Low', 'Close']:
        stock[col] = pd.to_numeric(stock[col], errors='coerce')
    stock = stock.set_index('Date').sort_index()

    entry_dt = pd.Timestamp(entry_date)
    exit_dt  = trade.exit_date
    idx      = stock.index
    ep       = idx.searchsorted(entry_dt)
    xp       = idx.searchsorted(exit_dt)
    start    = max(0, ep - window)
    end      = min(len(idx), xp + window + 1)
    sub      = stock.iloc[start:end]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 7),
        gridspec_kw={'height_ratios': [3, 1]}, sharex=True
    )

    ax1.plot(sub.index, sub['Close'], color='#1565C0', linewidth=1.5, label='收盤價')
    ax1.fill_between(sub.index, sub['Low'], sub['High'],
                     alpha=0.12, color='#1565C0', label='高低範圍')

    ax1.axvline(entry_dt, color='green', linewidth=1.5, linestyle='--', alpha=0.8)
    ax1.axvline(exit_dt,  color='red',   linewidth=1.5, linestyle='--', alpha=0.8)
    ax1.scatter([entry_dt], [trade.adj_entry_price],
                color='green', s=120, zorder=5,
                label=f'進場 {trade.adj_entry_price:.1f}')
    ax1.scatter([exit_dt], [trade.adj_exit_price],
                color='red', s=120, zorder=5, marker='v',
                label=f'出場 {trade.adj_exit_price:.1f}')

    pnl_color = '#2E7D32' if trade.pnl_net >= 0 else '#C62828'
    ax1.set_title(
        f"{ticker} ({trade.direction})  "
        f"進場: {entry_dt.date()} @ {trade.adj_entry_price:.1f}  "
        f"出場: {exit_dt.date()} @ {trade.adj_exit_price:.1f}  "
        f"持有: {trade.hold_days}天  "
        f"損益: {trade.pnl_net:+.0f}元  "
        f"原因: {trade.exit_reason}",
        color=pnl_color, fontsize=11
    )
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(alpha=0.3)
    ax1.set_ylabel('價格（元）')

    ax2.bar(sub.index, sub['Volume'], color='#90A4AE', alpha=0.8)
    ax2.set_ylabel('成交股數')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
%%time
# ── 修改這裡指定要分析哪幾筆交易 ──────────────────────────────────
# 格式：(股票代碼, 進場日期)，日期請與 trade_log.csv 裡的 entry_date 欄一致
trades_to_analyze = [
    ('1905', '2021-04-22'),
    ('5608', '2021-05-12'),
    ('2427', '2023-03-13'),
    ('5906', '2020-01-02'),
    ('6243', '2017-11-08'),
]
# ──────────────────────────────────────────────────────────────────

STOCKS_DIR = 'stocks_full'   # 個股 CSV 資料夾（相對於此 notebook 的位置）

for ticker, entry_date in trades_to_analyze:
    plot_trade(ticker, entry_date, df, stocks_dir=STOCKS_DIR)
